In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
import torch
from peft import PeftModel
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
lora= "meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_4_optim_adamw_torch_fused_r_64_alpha_128_lr_0.0002_drop_out_0.05_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
#lora = "meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_4_optim_ademamix_r_64_alpha_128_lr_0.0002_drop_out_0.05_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
model = PeftModel.from_pretrained(model, lora)


device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
prompt = "Мой кариотип 46,XX,t(19;13)(p11;q25). Что это значит??"
prompt = "Мой кариотип 46,XY . Что это значит?"
prompt = "Что такое ген"
prompt = "Мой кариотип 46,XY,der(21;21)(q10;q10),+21. Что это значит?"
inputs = tokenizer([prompt], return_tensors="pt").to(device)   # <-- вот так
model.past_key_values = None
streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer=streamer,
    max_new_tokens=1600,
    early_stopping=True,
    use_cache=False, 
    #no_repeat_ngram_size=2,
    #eos_token_id=tokenizer.eos_token_id,
)

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  
Кариотип: 46,XY,der(21;21)(q10;q10),+21
Тип хромосомной аномалии: структурно-числовая — транслокационная форма трисомии 21
Наименование синдрома: синдром Дауна, транслокационный вариант
Группа нарушений: аутосомные анеуплоидии, сбалансированные и несбалансированные Робертсоновские транслокации

1. Характеристика
Запись 46,XY,der(21;21)(q10;q10),+21 означает, что общее число хромосом — 46, однако присутствует деривативная хромосома der(21;21)(q10;q10) и дополнительная копия хромосомы 21.
Таким образом, генетический материал 21q представлен в трёх копиях (трисомия по 21). Короткие плечи акроцентрических хромосом утрачены и клинически незначимы.

2. Генетическая суть
Робертсоновские транслокации — это слияние длинных плеч двух акроцентрических хромосом (13, 14, 15, 21, 22) в области центромеры (q10;q10).
При наличии дополнительной хромосомы 21 формируется трисомия по 21q, что и определяет фенотип.

3. Фенотип и клинические проявления
Клиническая картина соответствует классической трисо

In [13]:
# Генерация с сохранением выхода
outputs = model.generate(
    **inputs,
    streamer=streamer,          # потоковый вывод в консоль (опционально)
    max_new_tokens=1600,
    early_stopping=True,
    use_cache=False,
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  
Кариотип: 46,XY,der(21;21)(q10;q10),+21
Тип хромосомной аномалии: структурно-числовая — транслокационная форма трисомии 21
Наименование синдрома: синдром Дауна, транслокационный вариант
Группа нарушений: аутосомные анеуплоидии, сбалансированные и несбалансированные Робертсоновские транслокации

1. Характеристика
Запись 46,XY,der(21;21)(q10;q10),+21 означает, что общее число хромосом — 46, однако присутствует деривативная хромосома der(21;21)(q10;q10) и дополнительная копия хромосомы 21.
Таким образом, генетический материал 21q представлен в трёх копиях (трисомия по 21). Короткие плечи акроцентрических хромосом утрачены и клинически незначимы.

2. Генетическая суть
Робертсоновские транслокации — это слияние длинных плеч двух акроцентрических хромосом (13, 14, 15, 21, 22) в области центромеры (q10;q10).
При наличии дополнительной хромосомы 21 формируется трисомия по 21q, что и определяет фенотип.

3. Фенотип и клинические проявления
Клиническая картина соответствует классической трисо

In [14]:

# Полный декодированный текст (включая исходный промпт)
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Если нужен только сгенерированный текст (без промпта):
input_length = inputs['input_ids'].shape[1]
generated_tokens = outputs[0][input_length:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

# Теперь переменная generated_text содержит чистый ответ модели
print("Сохранённый текст:", generated_text)

Сохранённый текст:   
Кариотип: 46,XY,der(21;21)(q10;q10),+21
Тип хромосомной аномалии: структурно-числовая — транслокационная форма трисомии 21
Наименование синдрома: синдром Дауна, транслокационный вариант
Группа нарушений: аутосомные анеуплоидии, сбалансированные и несбалансированные Робертсоновские транслокации

1. Характеристика
Запись 46,XY,der(21;21)(q10;q10),+21 означает, что общее число хромосом — 46, однако присутствует деривативная хромосома der(21;21)(q10;q10) и дополнительная копия хромосомы 21.
Таким образом, генетический материал 21q представлен в трёх копиях (трисомия по 21). Короткие плечи акроцентрических хромосом утрачены и клинически незначимы.

2. Генетическая суть
Робертсоновские транслокации — это слияние длинных плеч двух акроцентрических хромосом (13, 14, 15, 21, 22) в области центромеры (q10;q10).
При наличии дополнительной хромосомы 21 формируется трисомия по 21q, что и определяет фенотип.

3. Фенотип и клинические проявления
Клиническая картина соответствует

In [21]:
import pandas as pd 
df = pd.read_excel("ПРИМЕРЫ/данные_чистые.xlsx")
df = df.drop(columns = ['Unnamed: 0.1', 'Unnamed: 0'])
df

,Вопрос,Ответ,Теги,Класс
0,"Мой кариотип 46,XX,del(15)(q11q13). Что это зн...","Кариотип: 46,XX,del(15)(q11q13)\nТип хромосомн...",class_=del;chr=15;arm=q;region=15q11-q13;syndr...,del
1,"Мой кариотип 46,XY,del(15)(q11q13). Что это зн...","Кариотип: 46,XY,del(15)(q11q13)\nТип хромосомн...",class_=del;chr=15;arm=q;region=15q11-q13;syndr...,del
2,"Мой кариотип 46,XX,del(15)(q11q13). Что это зн...","Кариотип: 46,XX,del(15)(q11q13)\nТип хромосомн...",class_=del;chr=15;arm=q;region=15q11-q13;syndr...,del
3,"Мой кариотип 46,XY,del(15)(q11q13). Что это зн...","Кариотип: 46,XY,del(15)(q11q13)\nТип хромосомн...",class_=del;chr=15;arm=q;region=15q11-q13;syndr...,del
4,"Мой кариотип 46,XX,del(5)(p15.2). Что это значит?","Кариотип: 46,XX,del(5)(p15.2)\nТип хромосомной...",class_=del;chr=5;arm=p;region=known;syndrome=C...,del
...,...,...,...,...
1040,"Мой кариотип 47,XX,+21. В чём разница и что эт...","Кариотип: 47,XX,+21\nТип: полная (свободная) т...",Trisomy21;full;Down;aneuploidy,aneuploidy
1041,"Мой кариотип 47,XX,+21. В чём разница и что эт...","Кариотип: 47,XX,+21\nТип: полная (свободная) т...",Trisomy21;full;Down;aneuploidy,aneuploidy
1042,"Мой кариотип 47,XX,+21. В чём разница и что эт...","Кариотип: 47,XX,+21\nТип: полная (свободная) т...",Trisomy21;full;Down;aneuploidy,aneuploidy
1043,"Мой кариотип 47,XX,+21. В чём разница и что эт...","Кариотип: 47,XX,+21\nТип: полная (свободная) т...",Trisomy21;full;Down;aneuploidy,aneuploidy


In [26]:
df_10_percent = df.sample(frac=0.1, random_state=42)  # random_state для воспроизводимости
df_10_percent

,Вопрос,Ответ,Теги,Класс
773,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...,"class=t;sex_genetic=XX;chr_involved=9,13",t
280,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн...",class=inv; sex_genetic=XX; chr_involved=17; in...,inv
629,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом...",class=aneuploidy;type=trisomy;mosaic=yes;chr=8...,aneuploidy
318,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ...",aneuploidy;sex_chromosome;XXXXY;Klinefelter_va...,aneuploidy
1008,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес...",class=rob;sex_genetic=XY;chr_involved=22;homol...,rob
...,...,...,...,...
30,"Мой кариотип 46,XY,del(10)(q26.1q26.3). Что эт...","Кариотип: 46,XY,del(10)(q26.1q26.3)\nТип хромо...",class_=del;chr=10;arm=unknown;region=named;syn...,del
649,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...,"class=rob;sex_genetic=XY;chr_involved=21,22;ho...",rob
447,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома...",class=iso;chr=12;arm=p;syndrome=Pallister-Killian,iso
54,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а...",class_=del;chr=8;arm=p;region=p22;del_type=ter...,del


In [27]:
df_10_percent = df_10_percent.drop(columns = ['Теги', 'Класс']).reset_index(drop=True)
df_10_percent

,Вопрос,Ответ
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн..."
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом..."
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ..."
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес..."
...,...,...
99,"Мой кариотип 46,XY,del(10)(q26.1q26.3). Что эт...","Кариотип: 46,XY,del(10)(q26.1q26.3)\nТип хромо..."
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома..."
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а..."


In [28]:
def answering_model(s: str) -> str:

    prompt = s
    inputs = tokenizer([prompt], return_tensors="pt").to(device)   # <-- вот так
    model.past_key_values = None
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    
    outputs = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=1600,
        early_stopping=True,
        use_cache=False, 
        #no_repeat_ngram_size=2,
        #eos_token_id=tokenizer.eos_token_id,
    )
    # Полный декодированный текст (включая исходный промпт)
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Если нужен только сгенерированный текст (без промпта):
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return generated_text

In [30]:
answering_model("Мой кариотип 46,XY,der(21;21)(q10;q10),+21. Что это значит?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  
Кариотип: 46,XY,der(21;21)(q10;q10),+21
Тип хромосомной аномалии: структурно-числовая — транслокационная форма трисомии 21
Наименование синдрома: синдром Дауна, транслокационный вариант
Группа нарушений: аутосомные анеуплоидии, сбалансированные и несбалансированные Робертсоновские транслокации

1. Характеристика
Запись 46,XY,der(21;21)(q10;q10),+21 означает, что общее число хромосом — 46, однако присутствует деривативная хромосома der(21;21)(q10;q10) и дополнительная копия хромосомы 21.
Таким образом, генетический материал 21q представлен в трёх копиях (трисомия по 21). Короткие плечи акроцентрических хромосом утрачены и клинически незначимы.

2. Генетическая суть
Робертсоновские транслокации — это слияние длинных плеч двух акроцентрических хромосом (13, 14, 15, 21, 22) в области центромеры (q10;q10).
При наличии дополнительной хромосомы 21 формируется трисомия по 21q, что и определяет фенотип.

3. Фенотип и клинические проявления
Клиническая картина соответствует классической трисо

'  \nКариотип: 46,XY,der(21;21)(q10;q10),+21\nТип хромосомной аномалии: структурно-числовая — транслокационная форма трисомии 21\nНаименование синдрома: синдром Дауна, транслокационный вариант\nГруппа нарушений: аутосомные анеуплоидии, сбалансированные и несбалансированные Робертсоновские транслокации\n\n1. Характеристика\nЗапись 46,XY,der(21;21)(q10;q10),+21 означает, что общее число хромосом — 46, однако присутствует деривативная хромосома der(21;21)(q10;q10) и дополнительная копия хромосомы 21.\nТаким образом, генетический материал 21q представлен в трёх копиях (трисомия по 21). Короткие плечи акроцентрических хромосом утрачены и клинически незначимы.\n\n2. Генетическая суть\nРобертсоновские транслокации — это слияние длинных плеч двух акроцентрических хромосом (13, 14, 15, 21, 22) в области центромеры (q10;q10).\nПри наличии дополнительной хромосомы 21 формируется трисомия по 21q, что и определяет фенотип.\n\n3. Фенотип и клинические проявления\nКлиническая картина соответствует кл

In [31]:
df_10_percent["ответ ии"] = df_10_percent["Вопрос"].apply(answering_model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 Что ещё t(9;13) и t(13;13) называют?
Тип перестройки: сбалансированная реципрокная транслокация
1. Характеристика перестройки
У пациента выявлена реципрокная транслокация между хромосомами 9 и 13 с точками разрыва в областях p11 и q21 соответственно.
Это означает, что фрагмент хромосомы 9 (область 9p11) обменялся местами с фрагментом хромосомы 13 (область 13q21).
Поскольку материал не утрачен и не продублирован, кариотип сбалансированный — общее количество и состав генетического материала сохранены.

2. Фенотип носителя
У лиц со сбалансированными реципрокными транслокациями физическое и умственное развитие, как правило, нормальное.
Проблемы чаще проявляются в репродуктивной сфере (бесплодие/невынашивание), а не как системные клинические симптомы.

3. Репродуктивные последствия
У мужчин
• Возможна нормальная фертильность, однако часть сперматозоидов будет содержать несбалансированные хромосомные наборы из-за нарушенного спаривания хромосом в мейозе (формирование квадривалента).
• Возмо

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 Что изменения произошли в клетках? Что это значит для риска рождения ребёнка с синдромом Клайнфельтера?
Кариотип: 46,XX,inv(17)(q32q33)
Тип хромосомной аномалии: структурная — перицентрическая инверсия 17-й хромосомы
Группа нарушений: сбалансированные хромосомные перестройки

1. Характеристика
Запись 46,XX,inv(17)(q32q33) означает, что в одной из хромосом 17 произошёл переворот участка между точками q32 и q33.
Инверсия включает оба плеча хромосомы и затрагивает центромерную область.
Генетический материал не утрачен и не продублирован, изменён только порядок расположения сегмента, поэтому инверсия является сбалансированной.

2. Генетическая суть
Инверсии возникают при двух разрывах в одной хромосоме с последующим обратным воссоединением участка.
Носитель фенотипически здоров.
Однако при мейозе формируется инверсионная петля.
Если в пределах инвертированного сегмента происходит кроссинговер, возможны гаметы с дупликациями и делециями участка 17q32–17q33.

3. Фенотип носителя
• Физическо

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

 Что ещё изохромосома 12p?
А) Бархождения изохромосомы 12p как дополнительной хромосомы в клетках (sSMC / +12p).
Б) Генетический материал 12p представлен в трёх копиях (трисомия 12p).
К) Клинически значимый только вариант с сSMC (47,XX,+12p).

1. Что это значит?
47,XX,+i(12)(p10) означает, что у женщины вместо нормальных двух хромосом 12 присутствует ещё одна нормальная хромосома 12 и ещё дополнительная изохромосома 12p.
Таким образом, в клетках имеется:
•	Две копии длинного плеча хромосомы 12 (12q),
•	Две копии короткого плеча (12p) в составе другой изохромосомы.
Согласно ISCN (2020) и Gardner & Sutherland, Chromosome Abnormalities and Genetic Counseling (6th ed.), изохромосома 12p — частый структурный вариант синдрома Паллаистера–Киллиана (sSMC / +12p).

2. Что такое изохромосома 12p?
Изохромосома 12p — это морфологически изменённая форма хромосомы 12, встречающаяся примерно у 1:100 000 кариотипах и сопровождающаяся выраженным задержкой развития.
Основные клинические особенности:
•	В

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 Что ещё это синдром? Что особенности имеет этот вариант синдрома?
Кариотип: 46,XX,del(8)(p22)
Тип хромосомной аномалии: структурная — делеция
Группа: делеции (вариабельная клиническая значимость)

1. Характеристика
Утрата участка хромосомы может быть терминальной (потеря конца плеча) или интерстициальной (внутренний участок).
Клинический эффект зависит от размера делеции и того, какие дозочувствительные гены затронуты.

2. Возможные проявления
• задержка развития/речи,
• врождённые пороки (сердце, почки и др.) при вовлечении критических генов,
• нейроповеденческие особенности.
Иногда небольшие делецийные варианты могут быть доброкачественными, но это требует молекулярной оценки и интерпретации по базам.

3. Необходимые шаги
Рекомендуемые уточняющие исследования
• Хромосомный микромассив (array‑CGH / SNP‑массив) для точного определения размеров и генов в делеции.
• При необходимости — FISH/MLPA для подтверждения и семейного обследования.
• Кариотипирование родителей для оценки de novo/

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 Что ещё это синдром? Что особенности имеет этот вариант синдрома?

Кариотип: 46,XX,del(3)(q29q29)
Тип хромосомной аномалии: структурная — делеция (утрата участка хромосомы)
Наименование/подозрение: синдром Миллера–Дикера (делеция 3q29)
Группа нарушений: известные синдромы делеции

1. Характеристика
Делеция 3q29 (Миллер–Дикер синдром) — частая причина задержки развития среди делеций.

2. Клиника
• задержка психомоторного развития и речи,
• гипотония,
• судороги у части пациентов,
• кардиомиопатия/врождённые пороки сердца возможны,
• дисморфии, нарушения слуха/зрения.

3. Тактика
Рекомендуемые уточняющие исследования
• Хромосомный микромассив (array‑CGH / SNP‑массив) для точного определения размеров и генов в делеции.
• При необходимости — FISH/MLPA для подтверждения и семейного обследования.
• Кариотипирование родителей для оценки de novo/наследования и семейного риска.

Кардиолог (ЭхоКГ), невролог (ЭЭГ при судорогах), сурдолог/офтальмолог, реабилитация.

4. Заключение
Делеция 3q29 соо

In [32]:
df_10_percent.to_excel("ИРЕ_НА_ПРОВЕРКУ.xlsx")

In [33]:
df_10_percent

,Вопрос,Ответ,ответ ии
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...,Что ещё t(9;13) и t(13;13) называют?\nТип пер...
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн...",Что изменения произошли в клетках? Что это зн...
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом...",Что изменения произошли в клетках организма? ...
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ...",Что изменения произошли в клетках? Что это зн...
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес...",Что изменения произошли в клетках? Что это зн...
...,...,...,...
99,"Мой кариотип 46,XY,del(10)(q26.1q26.3). Что эт...","Кариотип: 46,XY,del(10)(q26.1q26.3)\nТип хромо...",Что ещё синдром «второй сердца»? Что это имее...
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...,Что изменения произошли в клетках?\nКариотип:...
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома...",Что ещё изохромосома 12p?\nА) Бархождения изо...
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а...",Что ещё это синдром? Что особенности имеет эт...
